In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()
model = ChatOpenAI()

In [5]:
#create a state
class BlogOutlineState(TypedDict):
    title: str
    outline: str
    content: str
    score: float

In [6]:
def generate_outline(state: BlogOutlineState) -> BlogOutlineState:
    title = state['title']
    prompt = f"Generate a blog outline for the title: {title}"
    outline = model.invoke(prompt).content
    state['outline'] = outline
    return state

In [7]:
def generate_content(state: BlogOutlineState) -> BlogOutlineState:
    outline = state['outline']
    prompt = f"Generate a blog content for the outline: {outline}"
    content = model.invoke(prompt).content
    state['content'] = content
    return state

In [13]:
def score_content(state: BlogOutlineState) -> BlogOutlineState:
    content = state['content']
    title = state['title']
    outline = state['outline']
    prompt = f"Score the blog content based on the title: {title} and outline: {outline}. Only give a score between 0 and 100."
    score = model.invoke(prompt).content
    state['score'] = score
    return state


In [11]:
#create a graph
graph = StateGraph(BlogOutlineState)

#add nodes
graph.add_node('generate_outline', generate_outline)
graph.add_node('generate_content', generate_content)
graph.add_node('score_content', score_content)

#add edges
graph.add_edge(START, 'generate_outline')
graph.add_edge('generate_outline', 'generate_content')
graph.add_edge('generate_content', 'score_content')
graph.add_edge('score_content', END)

workflow = graph.compile()

In [14]:
initial_state = {'title': 'AI and Machine Learning'}
final_state = workflow.invoke(initial_state)
print(final_state)

{'title': 'AI and Machine Learning', 'outline': "I. Introduction\n- Definition of AI and machine learning\n- Importance of AI and machine learning in today's technology landscape\n- Overview of how AI and machine learning are changing industries\n\nII. History of AI and Machine Learning\n- Origins of AI and machine learning \n- Milestones and key developments in the field \n- Impact of AI and machine learning on society\n\nIII. Applications of AI and Machine Learning\n- Use cases in various industries such as healthcare, finance, marketing, and more\n- Examples of successful AI and machine learning applications \n- Potential future applications and advancements\n\nIV. Challenges and Ethical Considerations\n- Data privacy and security concerns \n- Bias and fairness issues in AI algorithms\n- Potential job displacement and societal impacts\n\nV. The Future of AI and Machine Learning\n- Emerging trends in AI and machine learning\n- Opportunities for further development and innovation \n- 